# 📈 Predictive Analytics & Trend Forecasting
### Task 3: Building a Time-Series Regression model to forecast future revenue trends based on historical data.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

print("✅ Predictive modeling tools loaded successfully!")

### 1. Generating Historical Sales Data
We create 2 years of daily historical data (2024 to 2026) showing realistic upward growth trends mixed with holiday season spikes.

In [ ]:
np.random.seed(42)
dates = pd.date_range(start="2024-01-01", end="2026-05-31", freq="D")
n_days = len(dates)

# Create baseline growth trend + seasonal variance (spikes during holidays)
time_index = np.arange(n_days)
baseline_revenue = 5000 + (time_index * 4.5)  # Steady organic business growth
seasonality = 1200 * np.sin(2 * np.pi * dates.dayofyear / 365)  # Seasonal wave
noise = np.random.normal(0, 400, n_days)  # Daily random noise

historical_revenue = baseline_revenue + seasonality + noise

df_historical = pd.DataFrame({
    'Date': dates,
    'Historical_Revenue': historical_revenue
})

df_historical.tail(10)

### 2. Feature Engineering (Preparing Time for the AI Model)
Machine learning models cannot read raw calendar dates directly, so we transform dates into a numeric chronological sequence (`Day_Index`) that the model can learn from.

In [ ]:
df_historical['Day_Index'] = np.arange(len(df_historical))
X_train = df_historical[['Day_Index']]
y_train = df_historical['Historical_Revenue']
print("🛠️ Data engineered successfully. Ready for predictive training.")

### 3. Training the Trend Regression Model
We train a Linear Regression model to identify the overall growth vector of the business over time.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

# Evaluate model performance on historical baseline
predictions_historical = model.predict(X_train)
mae = mean_absolute_error(y_train, predictions_historical)
r2 = r2_score(y_train, predictions_historical)

print(f"📊 Model Evaluation Metrics:")
print(f"- Mean Absolute Error (MAE): ${mae:,.2f} variance per day")
print(f"- R-Squared Score (Fit Accuracy): {r2:.2f}")

### 4. Forecasting the Future (Next 30 Days)
We extend our calendar into the future by 30 days and use our trained model to forecast upcoming revenue trends.

In [ ]:
future_dates = pd.date_range(start="2026-06-01", end="2026-06-30", freq="D")
future_indices = np.arange(n_days, n_days + len(future_dates))

X_future = pd.DataFrame({'Day_Index': future_indices})
forecasted_revenue = model.predict(X_future)

df_forecast = pd.DataFrame({
    'Date': future_dates,
    'Forecasted_Revenue': forecasted_revenue
})

df_forecast.head(10)

### 5. Visualizing the Predictive Analytics Matrix
We stitch our historical realities and our AI's future predictions together on a clear line graph.

In [ ]:
fig = go.Figure()

# Plot past baseline data
fig.add_trace(go.Scatter(
    x=df_historical['Date'], y=df_historical['Historical_Revenue'],
    mode='lines', name='Historical Daily Revenue',
    line=dict(color='#3b82f6', width=1.5)
))

# Plot future predictions
fig.add_trace(go.Scatter(
    x=df_forecast['Date'], y=df_forecast['Forecasted_Revenue'],
    mode='lines+markers', name='AI 30-Day Forward Forecast',
    line=dict(color='#00df89', width=3, dash='dash')
))

fig.update_layout(
    title='🔮 Predictive Business Analytics: Revenue Forecasting Model',
    xaxis_title='Timeline',
    yaxis_title='Revenue ($)',
    template='plotly_dark',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)
fig.show()